# Data Cleaning

This notebook begins the cleaning workflow from the full merged item table. The first checkpoint loads the registered input, captures an in-memory baseline, and validates its schema and key/entity integrity before any cleaning step.

## 1. Locate the project

Run the notebook from the repository root or from `notebooks`. The root search also supports a working directory nested inside the repository.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def find_project_root(start_path: Path) -> Path:
    """Find the repository containing src/ and requirements.txt."""
    for path in [start_path, *start_path.parents]:
        if (path / "src").is_dir() and (path / "requirements.txt").is_file():
            return path
    raise FileNotFoundError(
        "Could not locate the project root containing src/ and requirements.txt."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from src.cleaning.validation import (
    assert_input_unchanged,
    capture_input_baseline,
    validate_cleaning_input,
)
from src.data_loader import load_data
from src.pipeline.contracts.schema import BENCHMARK_ROW_COUNT, FULL_MERGE_SCHEMA
from src.paths import INTERIM_DIR

print(f"Project root: {PROJECT_ROOT}")
print(f"Interim data path: {INTERIM_DIR}")
from src.cleaning.text import normalize_city_whitespace
from src.cleaning.measurements import clean_physical_measurements

## 2. Load the full merged table

Load `merged_order_items` from `data/interim` through the schema-aware project loader. Parquet is preferred, with CSV fallback; use `file_format="csv"` to select CSV directly.

In [ ]:
df = load_data("merged_order_items", stage="interim")

## 3. Capture the input baseline

Keep the baseline in memory. It records dimensions, column order, dtypes and row hashes that include the index. No second full-table copy or baseline file is created.

In [ ]:
input_baseline = capture_input_baseline(df)
input_summary = pd.DataFrame(
    [
        ["Item rows", BENCHMARK_ROW_COUNT, len(df)],
        ["Columns", len(FULL_MERGE_SCHEMA.columns), len(df.columns)],
    ],
    columns=["Metric", "Reference", "Observed"],
)
input_summary["Matches reference"] = input_summary["Reference"].eq(
    input_summary["Observed"]
)
display(input_summary)

## 4. Validate the cleaning input

Check exact columns and order, canonical dtypes, mandatory non-null identifiers, the unique item key, valid source entity keys and consistent repeated attributes. Repeated order, product, seller and customer IDs are expected at item grain; the profile reports them for context rather than treating them as duplicate items.

In [ ]:
input_validation = validate_cleaning_input(df)
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    print("Key profiles")
    display(input_validation["key_profile"])
    print("Entity consistency")
    display(input_validation["entity_consistency"])
    if not input_validation["conflicts"].empty:
        display(input_validation["conflicts"])

## 5. Confirm the input remains unchanged

This checkpoint performs validation only. Compare the table against the baseline before proceeding to a cleaning transformation.

In [ ]:
assert_input_unchanged(df, input_baseline)
print("Input validated and unchanged.")

## 6. Normalize city whitespace

Trim whitespace at the edges of `customer_city` and `seller_city`, and replace each internal run of whitespace with one regular space. Preserve case, spelling, missing values and all other columns. The function returns a clean copy and reports changes at item and entity grain.

In [ ]:
cleaned_df, text_change_summary = normalize_city_whitespace(df)
display(text_change_summary)

### Confirm the loaded source was preserved

The normalized result is held separately in `cleaned_df`. Recheck the loaded interim table against the baseline.

In [ ]:
assert_input_unchanged(df, input_baseline)
print("Loaded interim input remains unchanged.")

## 7. Handle non-positive physical measurements

Set observed weight or dimensions at or below zero to missing. Keep pre-existing missing measurements, freight values, and numeric tails unchanged. Flag affected item rows so downstream work can distinguish newly invalidated product measurements from values that were already missing.

In [ ]:
cleaned_df, measurement_summary = clean_physical_measurements(cleaned_df)
display(measurement_summary)
print(
    "Item rows flagged:",
    int(cleaned_df["has_invalid_physical_measurement"].sum()),
)

### Confirm the loaded source was preserved

Physical measurements are changed only in `cleaned_df`. Verify the original interim table again against its baseline.

In [ ]:
assert_input_unchanged(df, input_baseline)
print("Loaded interim input remains unchanged.")